# VIF SmartShelf — Notebook 00: Environment and Setup

**Purpose:** Validate the Colab runtime environment, install pinned dependencies, configure
directories and seeds, and produce a reproducibility manifest.

**Expected outcome:** All checks pass, dependencies installed, manifest saved.

**Prerequisites:** Google Colab with GPU runtime selected.

**Runtime selection:** `Runtime > Change runtime type > T4 GPU` (or A100 if available)

**Execution:** Run all cells top to bottom. Cells are designed to fail loudly if
requirements are not met. Do not skip cells.

---
**⚠️ Important:** This notebook installs packages and configures paths.
If you restart the runtime, you must re-run this notebook before running
any downstream notebook.
---

## 1. Python Version Check

In [ ]:
import sys

print(f"Python version: {sys.version}")
print(f"Python executable: {sys.executable}")

MIN_PYTHON = (3, 9)
if sys.version_info < MIN_PYTHON:
    raise RuntimeError(
        f"Python {MIN_PYTHON[0]}.{MIN_PYTHON[1]}+ required. "
        f"Current: {sys.version_info.major}.{sys.version_info.minor}. "
        "Change runtime type in Colab."
    )

print(f"✓ Python version OK ({sys.version_info.major}.{sys.version_info.minor})")

## 2. GPU Check

In [ ]:
import subprocess
import shutil

# Check if nvidia-smi is available
has_gpu = shutil.which("nvidia-smi") is not None

if has_gpu:
    result = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
         "--format=csv,noheader"],
        capture_output=True, text=True
    )
    gpu_info = result.stdout.strip()
    print(f"✓ GPU detected: {gpu_info}")
else:
    print("⚠️  No GPU detected. Training will run on CPU.")
    print("   For training: Runtime > Change runtime type > T4 GPU")
    print("   This notebook can still complete environment setup on CPU.")

## 3. Google Drive Mount

Mount Google Drive for persistent storage of datasets, checkpoints, and results.
Checkpoints survive Colab session resets when saved to Drive.

In [ ]:
import os

# Check if running in Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    print("✓ Google Drive mounted at /content/drive")
else:
    print("Not running in Colab — skipping Drive mount.")
    print("Running locally: set DRIVE_ROOT to your local storage path.")

## 4. Configuration — Edit These Variables

In [ ]:
# ================================================================
# PROJECT CONFIGURATION
# Edit these paths to match your Drive structure.
# ================================================================

# Root directory in Google Drive for all VIF SmartShelf files
DRIVE_ROOT = "/content/drive/MyDrive/vif-smartshelf"

# Subdirectories (created automatically below)
DATASET_DIR = f"{DRIVE_ROOT}/datasets"
CHECKPOINTS_DIR = f"{DRIVE_ROOT}/checkpoints"
ARTIFACTS_DIR = f"{DRIVE_ROOT}/artifacts"
REPORTS_DIR = f"{DRIVE_ROOT}/reports"
LOGS_DIR = f"{DRIVE_ROOT}/logs"

# Working directory in Colab (fast local SSD — use for active training)
COLAB_WORKDIR = "/content/smartshelf"

# Model settings
MODEL_FAMILY = "yolov8"  # yolov8 | yolov11 — set after checking Ultralytics version
DEFAULT_IMAGE_SIZE = 640
RANDOM_SEED = 42

print("Configuration:")
print(f"  DRIVE_ROOT       : {DRIVE_ROOT}")
print(f"  DATASET_DIR      : {DATASET_DIR}")
print(f"  CHECKPOINTS_DIR  : {CHECKPOINTS_DIR}")
print(f"  ARTIFACTS_DIR    : {ARTIFACTS_DIR}")
print(f"  REPORTS_DIR      : {REPORTS_DIR}")
print(f"  COLAB_WORKDIR    : {COLAB_WORKDIR}")
print(f"  MODEL_FAMILY     : {MODEL_FAMILY}")
print(f"  RANDOM_SEED      : {RANDOM_SEED}")

## 5. Create Directory Structure

In [ ]:
import pathlib

DIRS_TO_CREATE = [
    DATASET_DIR,
    f"{DATASET_DIR}/grocer-help",
    f"{DATASET_DIR}/roboflow-indian-grocery",
    CHECKPOINTS_DIR,
    ARTIFACTS_DIR,
    REPORTS_DIR,
    LOGS_DIR,
    COLAB_WORKDIR,
    f"{COLAB_WORKDIR}/data",
    f"{COLAB_WORKDIR}/runs",
]

for d in DIRS_TO_CREATE:
    pathlib.Path(d).mkdir(parents=True, exist_ok=True)

print("✓ Directory structure created:")
for d in DIRS_TO_CREATE:
    print(f"  {d}")

## 6. Disk Space Check

In [ ]:
import shutil

def check_disk_space(path, label, min_gb=5):
    """Check available disk space at path."""
    try:
        total, used, free = shutil.disk_usage(path)
        free_gb = free / (1024 ** 3)
        total_gb = total / (1024 ** 3)
        status = "✓" if free_gb >= min_gb else "⚠️"
        print(f"{status} {label}: {free_gb:.1f} GB free / {total_gb:.1f} GB total")
        if free_gb < min_gb:
            print(f"  WARNING: Less than {min_gb} GB free. May run out of space.")
        return free_gb
    except Exception as e:
        print(f"  Could not check {label}: {e}")
        return 0

# Colab local storage (~75 GB typical)
colab_free = check_disk_space("/content", "Colab local storage", min_gb=10)

# Google Drive
if IN_COLAB and os.path.exists("/content/drive"):
    drive_free = check_disk_space("/content/drive", "Google Drive", min_gb=5)

print()
print("Estimated disk requirements for Grocer-Help dataset:")
print("  Raw archive:           ~2–5 GB (estimated)")
print("  Extracted images:      ~5–15 GB (estimated — verify after download)")
print("  YOLOv8n training run:  ~500 MB per run")
print("  NOTE: Extract to Colab local storage for faster training I/O.")
print("  NOTE: Verify actual archive size before extracting.")

## 7. Install Dependencies

Install pinned versions for reproducibility. If versions change, update this cell
and document the reason in `docs/RISKS_AND_DECISIONS.md`.

In [ ]:
# Install with pinned versions
# Colab typically has torch pre-installed; ultralytics is the key addition

!pip install -q \
    "ultralytics>=8.3,<9.0" \
    "Pillow>=10.0,<12.0" \
    "opencv-python-headless>=4.9,<5.0" \
    "matplotlib>=3.8,<4.0" \
    "seaborn>=0.13,<0.14" \
    "pandas>=2.1,<3.0" \
    "numpy>=1.26,<2.0" \
    "tqdm>=4.66,<5.0" \
    "PyYAML>=6.0,<7.0" \
    "requests>=2.31,<3.0" \
    "huggingface_hub>=0.20,<1.0"

print("✓ Dependencies installed")

## 8. Verify PyTorch and Ultralytics

In [ ]:
import torch
import ultralytics
import numpy as np
import PIL
import cv2
import pandas as pd
import matplotlib

print("Package versions:")
print(f"  torch:       {torch.__version__}")
print(f"  ultralytics: {ultralytics.__version__}")
print(f"  numpy:       {np.__version__}")
print(f"  PIL:         {PIL.__version__}")
print(f"  cv2:         {cv2.__version__}")
print(f"  pandas:      {pd.__version__}")
print(f"  matplotlib:  {matplotlib.__version__}")

print()
print("PyTorch CUDA:")
print(f"  CUDA available:    {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  CUDA version:      {torch.version.cuda}")
    print(f"  GPU count:         {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {props.name}, {props.total_memory / 1e9:.1f} GB VRAM")
else:
    print("  No CUDA GPU available. Training will run on CPU (very slow).")
    print("  Action required: Runtime > Change runtime type > T4 GPU")

## 9. Configure Random Seeds

In [ ]:
import random
import os

def set_seeds(seed: int = 42):
    """Set random seeds for reproducibility.
    
    Note: Full reproducibility is not guaranteed across different GPU models,
    CUDA versions, or PyTorch versions. Seeds reduce (but don't eliminate) variance.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    print(f"✓ Random seeds set to {seed}")

set_seeds(RANDOM_SEED)

## 10. Verify Ultralytics YOLO (Quick Smoke Test)

In [ ]:
from ultralytics import YOLO

# Test that Ultralytics can load a pretrained model
# This downloads yolov8n.pt (~6MB) — a small pretrained COCO model
# We use this only to verify the framework works, not for VIF training
print("Loading YOLOv8n (smoke test — COCO pretrained, not VIF model)...")
test_model = YOLO("yolov8n.pt")

print(f"✓ Ultralytics YOLO loaded successfully")
print(f"  Model task:   {test_model.task}")
print(f"  Model type:   {type(test_model.model).__name__}")

# Run inference on a blank image (smoke test only)
import numpy as np
blank = np.zeros((640, 640, 3), dtype=np.uint8)
results = test_model(blank, verbose=False)
print(f"  Inference on blank image: {len(results[0].boxes)} detections (expected: 0)")
print("✓ Ultralytics inference pipeline works")

# Clean up
del test_model, results

## 11. Validate Dataset Source Access

Check network connectivity to dataset sources without downloading anything.

In [ ]:
import requests

SOURCES_TO_CHECK = [
    ("Zenodo (Grocer-Help record)", "https://zenodo.org/records/10464054"),
    ("Hugging Face Hub", "https://huggingface.co"),
    ("PyPI", "https://pypi.org"),
]

print("Checking network access to dataset sources:")
for name, url in SOURCES_TO_CHECK:
    try:
        resp = requests.head(url, timeout=10, allow_redirects=True)
        status = resp.status_code
        ok = status < 400
        icon = "✓" if ok else "⚠️"
        print(f"  {icon} {name}: HTTP {status}")
    except requests.RequestException as e:
        print(f"  ✗ {name}: Connection failed — {e}")

print()
print("Note: HTTP 200 only confirms the URL is reachable.")
print("Grocer-Help file access still requires Zenodo login — handled in Notebook 01.")

## 12. Save Reproducibility Manifest

In [ ]:
import json
import datetime
import platform

manifest = {
    "generated_at": datetime.datetime.utcnow().isoformat() + "Z",
    "notebook": "00_environment_and_setup.ipynb",
    "environment": {
        "python_version": sys.version,
        "platform": platform.platform(),
        "in_colab": IN_COLAB,
    },
    "gpu": {
        "available": torch.cuda.is_available(),
        "count": torch.cuda.device_count() if torch.cuda.is_available() else 0,
        "devices": [
            {
                "index": i,
                "name": torch.cuda.get_device_properties(i).name,
                "memory_gb": round(torch.cuda.get_device_properties(i).total_memory / 1e9, 1)
            }
            for i in range(torch.cuda.device_count())
        ] if torch.cuda.is_available() else [],
    },
    "packages": {
        "torch": torch.__version__,
        "ultralytics": ultralytics.__version__,
        "numpy": np.__version__,
        "pillow": PIL.__version__,
        "opencv": cv2.__version__,
        "pandas": pd.__version__,
        "matplotlib": matplotlib.__version__,
    },
    "configuration": {
        "drive_root": DRIVE_ROOT,
        "dataset_dir": DATASET_DIR,
        "checkpoints_dir": CHECKPOINTS_DIR,
        "artifacts_dir": ARTIFACTS_DIR,
        "random_seed": RANDOM_SEED,
        "default_image_size": DEFAULT_IMAGE_SIZE,
        "model_family": MODEL_FAMILY,
    }
}

manifest_path = f"{REPORTS_DIR}/environment_manifest.json"
with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)

print(f"✓ Reproducibility manifest saved to: {manifest_path}")
print()
print("Summary:")
print(json.dumps(manifest, indent=2))

## 13. Final Environment Check

If this cell passes, the environment is ready for Notebook 01.

In [ ]:
checks = [
    ("Python 3.9+", sys.version_info >= (3, 9)),
    ("PyTorch importable", True),  # would have failed above if not
    ("Ultralytics importable", True),
    ("Dataset directory exists", os.path.isdir(DATASET_DIR)),
    ("Checkpoints directory exists", os.path.isdir(CHECKPOINTS_DIR)),
    ("Reports directory exists", os.path.isdir(REPORTS_DIR)),
    ("Manifest saved", os.path.isfile(manifest_path)),
]

print("Final checks:")
all_passed = True
for name, result in checks:
    icon = "✓" if result else "✗"
    print(f"  {icon} {name}")
    if not result:
        all_passed = False

print()
if all_passed:
    print("✅ All checks passed. Environment ready.")
    print("   Next: Run Notebook 01 to acquire and verify the dataset.")
    print()
    print("   IMPORTANT: Grocer-Help requires Zenodo account access.")
    print("   If access has not been requested yet, do so before running Notebook 01.")
    print("   URL: https://zenodo.org/records/10464054")
else:
    raise RuntimeError(
        "Some checks failed. Resolve the issues above before proceeding."
    )